In [1]:
import sys
from pathlib import Path

path = Path().cwd().parent / "src"
sys.path.insert(0, str(path))

In [2]:
from dask_obj.expr import *
from dask_obj.core import *

In [3]:
from collections import deque

In [4]:
import operator

In [5]:
from copy import deepcopy

In [6]:
from dask import compute, delayed, persist

In [7]:
import pandas as pd
import numpy as np
import polars as pl
import toolz

In [8]:
from dask_obj.expr import OldExpr

In [9]:
o = OldExpr("o", 1)
str(o)

'o = (1)'

In [10]:
from typing import Callable

from dask_obj.expr import _getattr, _hasattr, op
from boltons.funcutils import format_invocation

In [11]:
>>> e = Expr("e")
>>> print(e)
>>> print(e.foo)
>>> print(e.foo(1, 2, 3))
>>> print(e.foo(1, 2, 3).bar)
>>> print(e.foo(1, 2, 3).bar(4, 5, 6))
>>> print(e.foo(1, 2, 3).bar.baz)
>>> print(e.F(str.upper))
>>> print(e.F(str.upper).lower())
>>> print(e.F(str.upper).lower().F(str.title))
>>> print(e.lower().F(str.upper))
>>> print(e.F(str.upper).F(str.title).lower())

e
e.foo
e.foo(1, 2, 3)
e.foo(1, 2, 3).bar
e.foo(1, 2, 3).bar(4, 5, 6)
e.foo(1, 2, 3).bar.baz
str.upper(e)
str.upper(e).lower()
str.title(str.upper(e).lower())
str.upper(e.lower())
str.title(str.upper(e)).lower()


In [12]:
e = Expr(10)
e + 1

(10).__add__(1)

In [13]:
(10).__add__(1)

11

In [14]:
hasattr(Expr("", None), "obj__")

True

In [15]:
e = Expr("e", 10)
e

e

In [16]:
e = Expr("e")
e = e.F(str.upper).lower().F(str.title)
print(e)
print(e.eval())

str.title(str.upper(e).lower())
E


In [17]:
list(map(e.eval, ["abc", "def", "ghi"]))

['Abc', 'Def', 'Ghi']

In [18]:
e = -Expr(10) + 12
print(e)
e

(10).__neg__().__add__(12)


(10).__neg__().__add__(12)

In [19]:
result = e.eval()
print(f"{result=}")

result=2


In [20]:
f = e.F(operator.add, 100)
print(f)
print(repr(f))
f.eval()

add((10).__neg__().__add__(12), 100)
add((10).__neg__().__add__(12), 100)


102

In [21]:
list(map(e.eval, range(10)))

[12, 11, 10, 9, 8, 7, 6, 5, 4, 3]

In [22]:
e = Expr(pd.DataFrame, columns=["a", "b"], index=[1, 2, 3]).fillna(0).pipe(lambda df: df.sort_values("a")).head(2)
print(e)
print(repr(e))
e.eval()

(<class 'pandas.core.frame.DataFrame'>).fillna(0).pipe(<function <lambda> at 0x7faff4058860>).head(2)
(<class 'pandas.core.frame.DataFrame'>).fillna(0).pipe(<function <lambda> at 0x7faff4058860>).head(2)


,a,b
1,0,0
2,0,0


In [23]:
a = getattr(Expr(int), "__add__")
print(a)
str(a), type(a), callable(a), a.__name__

<bound method Expr.make_op.<locals>.func of (<class 'int'>)>


("<bound method Expr.make_op.<locals>.func of (<class 'int'>)>",
 method,
 True,
 '__add__')

In [24]:
a = a(1)
str(a), type(a), callable(a), a

("(<class 'int'>).__add__(1)",
 dask_obj.expr.Expr,
 True,
 (<class 'int'>).__add__(1))

In [25]:
l = Expr(list("abc"))
print(l)
l1 = l[1]
print(l1)
l1.eval()

(['a', 'b', 'c'])
(['a', 'b', 'c']).__getitem__(1)


'b'

In [26]:
BaseExpr = Expr

In [27]:
class Expr(BaseExpr):
    def map(self, it):
        return map(self.eval, it)

In [28]:
class Expr(BaseExpr):
    def map(self, it):
        eval = delayed(self.eval)
        return list(map(eval, it))

In [29]:
e = -(Expr(int) + 1) ** 3
print(e)
print(e.eval(-5))
list(map(e.eval, range(10)))

(<class 'int'>).__add__(1).__pow__(3).__neg__()
64


[-1, -8, -27, -64, -125, -216, -343, -512, -729, -1000]

In [30]:
compute(e.map(range(10)))

([-1, -8, -27, -64, -125, -216, -343, -512, -729, -1000],)

In [31]:
class Expr(BaseExpr):
    def map(self, it):
        return list(map(self.eval, it))

In [32]:
e = -(Expr(int) + 1) ** 3
print(e)
print(e.eval(-5))
list(map(e.eval, range(10)))

(<class 'int'>).__add__(1).__pow__(3).__neg__()
64


[-1, -8, -27, -64, -125, -216, -343, -512, -729, -1000]

In [33]:
e.map(range(10))

[-1, -8, -27, -64, -125, -216, -343, -512, -729, -1000]

In [34]:
list(map(lambda _: (_).__add__(1).__pow__(3).__neg__(), range(0, 10)))

[-1, -8, -27, -64, -125, -216, -343, -512, -729, -1000]

In [35]:
f = Expr(range(13)).F(toolz.flip(map), e.eval).F(list)
f.eval()

[-1, -8, -27, -64, -125, -216, -343, -512, -729, -1000, -1331, -1728, -2197]

In [36]:
def get_root_value(expr):
    while expr.expr__ is not None:
        expr = expr.expr__
    return expr.obj__

get_root_value(f)

range(0, 13)

In [37]:
def get_root_expr(expr):
    while expr.expr__ is not None:
        expr = expr.expr__
    return expr

root = get_root_expr(e)
print(type(root))
print(root)
print(root.expr__)

<class '__main__.Expr'>
(<class 'int'>)
None


In [38]:
def reduce_expr(expr):
    exprs = deque()
    while expr.expr__ is not None:
        exprs.insert(0, (expr.obj__, expr.args__, expr.kw__))
        expr = expr.expr__
    exprs.insert(0, (expr.obj__, expr.args__, expr.kw__))
    return tuple(exprs)

def expr_maker(exprs):
    expr = None
    for obj, args, kw in exprs:
        expr = Expr(obj, *args, **kw, expr=expr)
    return expr

expr_maker(reduce_expr(e))

(<class 'int'>).__add__(1).__pow__(3).__neg__()

In [39]:
def replace_root_value(expr, value, *args, **kw):
    _, *exprs = reduce_expr(expr)
    exprs = ((value, args, kw), *exprs)
    return expr_maker(exprs)

print(e)
new = replace_root_value(e, 10)
print(e)
print(new)
print(new.eval())

(<class 'int'>).__add__(1).__pow__(3).__neg__()
(<class 'int'>).__add__(1).__pow__(3).__neg__()
(10).__add__(1).__pow__(3).__neg__()
-1331


In [40]:
e = (((((-Expr(int)) + 1) * 3) + 1) / 2).F(int)
print(e)

int((<class 'int'>).__neg__().__add__(1).__mul__(3).__add__(1).__truediv__(2))


In [41]:
objs = DaskDelayedObjects(range(10))
objs

<class 'dask_obj.core.DaskDelayedObjects'>([Delayed('noop-55446534-f858-4373-9631-d62b6bff2fcd'), Delayed('noop-e5302081-a390-402c-8247-50f07d8d9b37'), Delayed('noop-b50b4e94-b629-4777-afb3-0bdc5233729a'), Delayed('noop-09c73ed9-794a-44ee-9c35-a6bc3a09ae9b'), Delayed('noop-8ac715b5-a780-4447-9b00-41d4960ddc82'), Delayed('noop-cc3bd9a0-d49b-40fb-a4db-a39cd5cf9026'), Delayed('noop-6ee93adf-a751-4521-aeef-3f7fe7ff3061'), Delayed('noop-4329aacd-0c2b-4f74-9d71-d9848e138d85'), Delayed('noop-414b95d4-7559-4172-a069-94e0960c03b7'), Delayed('noop-9b6f593f-b7d6-4283-97d8-35b8233230a5')])

In [42]:
new = objs.map(e.eval)
print(new)
new.compute()

<class 'dask_obj.core.DaskDelayedObjects'>([Delayed('eval-fab74b24-356d-45cc-b814-cf096d73ce1f'), Delayed('eval-42313ef5-468c-4f1f-b39f-2c301c264e69'), Delayed('eval-9246f4cd-1dee-46a8-bda7-12cb2283718e'), Delayed('eval-a39c72bd-5464-4c37-b00d-3f6bdc0077e4'), Delayed('eval-796d9d4f-fae1-427c-98b1-04d83628089b'), Delayed('eval-0ea62d97-18d2-4a53-9f82-20790658956b'), Delayed('eval-a3fa17e3-4c34-4c22-81fe-c511a0ca771c'), Delayed('eval-c6d1aebd-5322-436b-9702-717fc93e8eb3'), Delayed('eval-a103f20e-ee7e-4e4a-ade2-2c11232fc307'), Delayed('eval-be7c036b-cd57-4ea5-b077-8bead798faa8')])


[2, 0, -1, -2, -4, -5, -7, -8, -10, -11]

In [43]:
objs.map(Expr(int).F(range).F(list).eval).compute()

[[],
 [0],
 [0, 1],
 [0, 1, 2],
 [0, 1, 2, 3],
 [0, 1, 2, 3, 4],
 [0, 1, 2, 3, 4, 5],
 [0, 1, 2, 3, 4, 5, 6],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7, 8]]

In [44]:
from dask.delayed import Delayed

In [45]:
items = list(map(noop, range(10)))
print(type(items[0]))
isinstance(items[0], Delayed)

<class 'int'>


False

In [46]:
DaskDelayedObjects(list(map(Expr, range(10))))

<class 'dask_obj.core.DaskDelayedObjects'>([Delayed('noop-17785545-6fdd-49d1-b337-f2910ea4f79a'), Delayed('noop-1374c694-e7c6-4ee6-b4c4-03b35a5c8c76'), Delayed('noop-1454cc35-2d79-4a64-b435-09a2c9c0c366'), Delayed('noop-5dd55c15-8e97-493a-bd86-bdf2cc7ab0de'), Delayed('noop-44fb384b-71f7-4020-9c7d-9def9acc1220'), Delayed('noop-8e763e94-2a6b-4da3-80cb-950a977bac4f'), Delayed('noop-b7bb7e21-b218-4593-9563-68103e94c2d5'), Delayed('noop-4c71945d-5ce7-4586-a4f3-f6b884dfae1e'), Delayed('noop-f32756c1-481c-4519-b45f-09953571f8ea'), Delayed('noop-0ea02d96-f15b-462c-8240-fa2b127debfe')])

In [52]:
objs = DaskDelayedObjects(map(Expr, range(10)))

In [53]:
(objs + 10).eval().compute()

[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]